# 기업 주력상품 HS + 향후 수출 전망 (+ 품목설명)

이 노트북 **하나만 위에서부터 끝까지 실행**하면, 기업별 주력상품 HS 코드에 **품목설명**과 **향후 수출 전망 증가율**이 붙은 표가 나옵니다.
설정할 건 ⑤의 `BASE_DATE_OVERRIDE` 하나뿐(비우면 자동).

In [1]:
import sys
from pathlib import Path
try:
    current_path = Path(__file__).resolve()
except NameError:
    current_path = Path().resolve()
for parent in current_path.parents:
    if (parent / "stock_forecast" / "DATA").is_dir():
        stock_forecast_path = parent / "stock_forecast"; break
else:
    raise ImportError("stock_forecast/DATA 폴더를 찾을 수 없습니다.")
if str(stock_forecast_path) not in sys.path:
    sys.path.insert(0, str(stock_forecast_path))
print("sys.path 등록:", stock_forecast_path)

sys.path 등록: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\stock_forecast


In [2]:
import re
import pandas as pd
from DATA.stock_invest_function import *

db_info = {'user':'stox7412','password':'Apt106503!~','host':get_db_host(),'port':'3307','database':'investar'}

def normalize_hs6(x):
    """HS 코드를 6자리로 통일 (실수형 표기/선행 0 유실 자동 복원)."""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    s = re.sub(r"\D", "", s)
    if not s:
        return None
    if len(s) % 2 == 1:
        s = "0" + s
    if len(s) < 6:
        return None
    return s[:6]

## HS 코드 → 품목 설명 사전 (Claude 지식 기반)
대화에 나온 코드 + 흔한 중소형 수출품목 + 예시(MLCC, 치과 X-ray)를 담았습니다.
**사전에 없는 코드는 HS 류(2자리) 단위로 자동 설명**됩니다. 새 품목은 `HS6_DESC`에 한 줄 추가하면 됩니다.
※ HS는 2017/2022 개정에 따라 일부 세번이 다를 수 있으니 핵심 종목은 한 번 검증 권장.

In [3]:
HS6_DESC = {
 # 반도체·전자부품
 "854232":"메모리 반도체 (DRAM·HBM 등)", "854231":"시스템반도체 - 프로세서·컨트롤러(CPU/MCU)",
 "854233":"시스템반도체 - 증폭기 IC", "852351":"SSD·플래시메모리 등 반도체 저장장치",
 "847330":"컴퓨터(자동자료처리기계) 부품·부속", "852990":"방송·영상·카메라기기 부품(카메라모듈·안테나 등)",
 "848620":"반도체·디스플레이 제조장비", "841410":"진공펌프",
 "853224":"적층세라믹콘덴서 (MLCC)", "853221":"탄탈 콘덴서", "853210":"전력용 콘덴서(역률보상)",
 "854411":"동 권선(에나멜 동선)", "854519":"탄소·흑연 전극", "850511":"영구자석(금속, NdFeB 등)",
 "851762":"통신·네트워크 장비(송수신·중계기)", "741021":"동박(PCB·배터리용, 적층)",
 # 배터리·2차전지
 "850760":"리튬이온 축전지(배터리 셀)", "850790":"축전지(배터리) 부분품",
 "282520":"리튬 산화물·수산화물(양극재 원료)", "284190":"금속산 염(기타)-리튬금속산화물(양극활물질) 등",
 "760711":"알루미늄박(배터리·포장용)",
 # 광·태양광·LED
 "854141":"발광다이오드(LED)", "854142":"태양전지(셀, 미조립)", "854143":"태양광 모듈·패널",
 # 전력기기
 "853590":"고압 개폐·보호기기(>1000V)", "853720":"고압 배전·제어반(>1000V)",
 "853620":"배선용 회로차단기(<=1000V)", "853890":"개폐·배전기기 부품",
 "850440":"정지형 변환기(인버터·충전기·SMPS)", "850434":"대형 변압기(>500kVA, 초고압 변압기)",
 "851150":"차량용 발전기(올터네이터)·점화기기",
 # 방산
 "930591":"군용 무기 부분품·부속품", "852610":"레이더 장치",
 # 조선·운송장비·자동차
 "890120":"탱커(유조선)", "860310":"전기식 철도 동차(자가추진)", "860900":"운송용 컨테이너",
 "870590":"특수목적 자동차(기타)", "870880":"자동차 서스펜션 시스템·부품", "870829":"자동차 차체 부품(기타)",
 "840820":"차량용 디젤엔진", "841229":"유압식 엔진·모터(기타)", "842123":"엔진용 오일·연료 여과기",
 "401110":"승용차용 타이어(신품)",
 # 기계·공구·금형
 "847982":"혼합·반죽 기계(2차전지 슬러리 등)", "847981":"금속 처리용 기계(권선기 등)",
 "847950":"산업용 로봇(기타)", "844399":"인쇄기 부품·부속", "842489":"분사·살포 기계기구(기타)",
 "848071":"사출·압축 금형(플라스틱·고무용)", "820770":"밀링용 교환 절삭공구",
 "820900":"초경 인서트 팁(공구용)", "680421":"합성다이아몬드 연삭·연마석",
 # 석유화학·플라스틱·고무·화학
 "271019":"석유제품(경유·등유·중유 등, 원유 제외)", "290124":"부타디엔·이소프렌",
 "292910":"이소시아네이트(MDI·TDI)", "390330":"ABS 수지", "390320":"SAN 수지",
 "390290":"폴리프로필렌계 중합체(기타)", "390220":"폴리이소부틸렌", "390130":"에틸렌-비닐아세테이트(EVA)",
 "390761":"PET 수지", "390730":"에폭시 수지", "390799":"폴리에스테르(기타,1차형태)",
 "390690":"아크릴 중합체(기타)", "390390":"스티렌계 중합체(기타)", "391110":"석유수지·쿠마론수지",
 "391190":"석유수지 등(기타,1차형태)", "392190":"플라스틱 판·시트·필름(기타)", "392690":"플라스틱 제품(기타)",
 "400251":"합성고무-NBR 라텍스(니트릴 라텍스)", "400219":"스티렌부타디엔고무(SBR,기타)", "400220":"부타디엔고무(BR)",
 "280700":"황산", "250300":"황", "381700":"혼합 알킬벤젠(LAB, 세제원료)", "382600":"바이오디젤",
 "291739":"방향족 다가카르복실산(기타)", "293499":"핵산·헤테로고리화합물(의약 중간체 등)",
 # 화장품·생활화학
 "330499":"기초·색조 화장품(기타)", "330790":"향료·화장용 제품(기타, 탈취제 등)",
 "340130":"피부세정용 계면활성 제품(바디워시·클렌저)",
 # 바이오·의료
 "300214":"면역제품(백신·항체 등, 소매포장)", "300490":"의약품(기타, 소매용)",
 "901890":"의료·외과용 기기(기타)", "902213":"치과용 X선 장치(Dental X-ray)",
 "902214":"의료·수의용 X선 장치", "902212":"컴퓨터단층촬영기(CT)",
 # 철강·비철금속
 "730890":"철강 구조물·부분품", "721420":"철·비합금강 봉", "721499":"철강 봉(기타)", "721633":"H형강",
 "720836":"광폭 열연강판(코일)", "720712":"철강 반제품(빌릿·슬래브)", "730629":"유정용 케이싱·튜빙 강관(기타)",
 "730630":"용접 철강관(원형단면,기타)", "730511":"송유관용 대구경 강관(SAW)",
 "730729":"스테인리스 관이음쇠(기타)", "730793":"철강 맞대기용접 관이음쇠",
 "720270":"페로몰리브덴", "720260":"페로니켈",
 "740311":"전기동-정련구리 캐소드", "740200":"미정련 구리(조동)", "740329":"구리합금 잉곳(기타)",
 "740400":"구리 스크랩", "740819":"정련 구리선(기타)", "740811":"정련 구리선(>6mm)",
 "740911":"정련 구리 판·대(코일)", "740921":"황동 판·대(코일)", "740721":"황동 봉·형재",
 "760200":"알루미늄 스크랩", "760120":"알루미늄 합금(미가공)", "760612":"알루미늄합금 판·대",
 "760429":"알루미늄 형재·봉(기타)", "761699":"알루미늄 제품(기타)", "261310":"몰리브덴광(배소)",
 # 귀금속 (가격효과 주의)
 "710691":"은(미가공)", "710692":"은(반제품)", "710813":"금(반제품)", "711011":"백금(미가공·분말)",
 "711019":"백금(반제품)", "711021":"팔라듐(미가공·반제품)", "711292":"백금 스크랩",
 "711319":"귀금속 장신구(기타)", "284329":"은 화합물",
 # 식품·수산
 "190230":"파스타·라면류(조리 안 된 기타)", "030354":"냉동 고등어", "030389":"냉동 어류(기타)",
 "030487":"냉동 참치 필레",
 # 기타
 "481190":"도포·가공 종이(기타)", "490199":"인쇄 서적(기타)", "610910":"면 티셔츠(편물)",
 "690919":"이화학용 세라믹 제품(기타)", "851690":"전열기기 부품", "852349":"광학기록매체(기타)",
 "320730":"액체 광택제(도자기·유리용)", "901320":"레이저 기기(기타)", "903090":"전기 측정·검사기기 부품",
 "903190":"측정·검사기기 부품(기타)", "903149":"광학식 측정·검사기기(기타)",
}

HS_CHAPTER = {
 "03":"어패류","07":"채소","08":"과일·견과","09":"커피·차·향신료","15":"동식물성 유지",
 "16":"육·어류 조제품","17":"당류","19":"곡물·곡분 조제품(면류 등)","20":"채소·과일 조제품",
 "21":"기타 조제식료품","22":"음료·주류","23":"사료","24":"담배","25":"소금·황·토석","26":"광·슬래그",
 "27":"광물성 연료(석유)","28":"무기화학품","29":"유기화학품","30":"의료용품","31":"비료","32":"염료·도료",
 "33":"화장품·향료","34":"비누·계면활성제","35":"단백질·접착제·효소","38":"각종 화학공업품","39":"플라스틱",
 "40":"고무","44":"목재","47":"펄프","48":"지·판지","49":"인쇄물","52":"면","54":"인조필라멘트",
 "55":"인조스테이플","59":"공업용 직물","60":"편물","61":"의류(편물)","62":"의류(직물)","63":"기타 섬유제품",
 "68":"석·시멘트 제품","69":"도자제품","70":"유리","71":"귀금속·보석","72":"철강","73":"철강 제품",
 "74":"구리","75":"니켈","76":"알루미늄","78":"납","79":"아연","80":"주석","81":"기타 비금속",
 "82":"공구·날붙이","83":"각종 비금속 제품","84":"일반기계","85":"전기·전자기기","86":"철도차량",
 "87":"자동차·부품","88":"항공기","89":"선박","90":"광학·의료·정밀기기","91":"시계","94":"가구·조명",
 "95":"완구·운동용구","96":"잡품",
}

def hs_desc(code6):
    if pd.isna(code6):
        return ""
    if code6 in HS6_DESC:
        return HS6_DESC[code6]
    ch = str(code6)[:2]
    if ch in HS_CHAPTER:
        return f"({ch}류) {HS_CHAPTER[ch]}"
    return ""

print("HS6 설명 사전:", len(HS6_DESC), "건 | 류 폴백:", len(HS_CHAPTER), "개")

HS6 설명 사전: 139 건 | 류 폴백: 65 개


In [4]:
# ⑤ 유일한 설정 ───────────────────────────────
BASE_DATE_OVERRIDE = None   # 예: "2025-06-30". None이면 최신 전망 기준일 자동
# ──────────────────────────────────────────────

In [5]:
# 향후 수출 전망 증가율 (HS6별)
fc = fetch_table_data(db_info, 'korea_monthly_trade_data_forecast')
fc['date'] = pd.to_datetime(fc['date'])
fc = fc.dropna(subset=['expDlr_forecast_12m']).sort_values(['root_hs_code','date'])
g = fc.groupby('root_hs_code')['expDlr_forecast_12m']
fc['past_12m']   = g.transform(lambda x: x.rolling(12, min_periods=12).sum())
fc['future_12m'] = g.transform(lambda x: x.shift(-11).rolling(12, min_periods=12).sum())
fc['수출전망증가율(%)'] = ((fc['future_12m']/fc['past_12m'] - 1)*100).round(2)

valid = fc.dropna(subset=['past_12m','future_12m'])
BASE_DATE = pd.to_datetime(BASE_DATE_OVERRIDE) if BASE_DATE_OVERRIDE else valid['date'].max()
print("전망 기준일:", BASE_DATE.date())

growth6 = valid.loc[valid['date']==BASE_DATE, ['root_hs_code','past_12m','future_12m','수출전망증가율(%)']].copy()
growth6['hscode_6d'] = growth6['root_hs_code'].map(normalize_hs6)
growth6 = growth6.dropna(subset=['hscode_6d']).drop_duplicates('hscode_6d')
print("전망 HS6 개수:", len(growth6))

✅ 'korea_monthly_trade_data_forecast' 테이블에서 241385건의 데이터를 가져왔습니다.
전망 기준일: 2025-10-31
전망 HS6 개수: 266


In [6]:
# 기업-HS 맵 로드 + 6자리 통일
manual_entries = [
    # ('A031330', '에스에이엠티', '854232'),
]
comp_raw = fetch_table_data(db_info, 'korea_company_hscode_map')

def _pick(cols, cands):
    low = {str(c).lower(): c for c in cols}
    for cand in cands:
        if cand.lower() in low:
            return low[cand.lower()]
    return None

col_hs   = _pick(comp_raw.columns, ['hs_code','hscode','hs','root_hs_code','hs_code_6d'])
col_name = _pick(comp_raw.columns, ['Name','company','company_name','기업명','종목명'])
col_tkr  = _pick(comp_raw.columns, ['ticker','Code','code','종목코드'])
assert col_hs and col_name, f"hs_code/Name 컬럼 감지 실패: {list(comp_raw.columns)}"

ren = {col_hs:'hs_code_raw', col_name:'Name'}
if col_tkr: ren[col_tkr] = 'ticker'
comp = comp_raw[[c for c in [col_tkr,col_name,col_hs] if c]].rename(columns=ren).copy()
if manual_entries:
    comp = pd.concat([comp, pd.DataFrame(manual_entries, columns=['ticker','Name','hs_code_raw'])], ignore_index=True)
comp['hscode_6d'] = comp['hs_code_raw'].map(normalize_hs6)
comp = comp.dropna(subset=['hscode_6d']).reset_index(drop=True)
print("기업-HS 행수:", len(comp), "| 기업수:", comp['Name'].nunique())

✅ 'korea_company_hscode_map' 테이블에서 762건의 데이터를 가져왔습니다.
기업-HS 행수: 622 | 기업수: 410


In [10]:
comp

,ticker,Name,hs_code_raw,hscode_6d
0,A093370,후성,854321,854321
1,A036490,SK머티리얼즈,281290,281290
2,A104830,원익머트리얼즈,854239,854239
3,A144960,뉴파워프라즈마,854239,854239
4,A036930,주성엔니지어링,847989,847989
...,...,...,...,...
617,A011784,금호석유,4002590000,400259
618,A011785,금호석유,4002110000,400211
619,A178920,PI첨단소재,3916909000,391690
620,A000070,삼양홀딩스,290723,290723


In [7]:
# 결합: 기업 주력상품 + 품목설명 + 향후 수출 전망
final = comp.merge(growth6[['hscode_6d','past_12m','future_12m','수출전망증가율(%)']], on='hscode_6d', how='left')
final['품목설명'] = final['hscode_6d'].map(hs_desc)

cols = [c for c in ['ticker','Name','hs_code_raw','hscode_6d','품목설명',
                    'past_12m','future_12m','수출전망증가율(%)'] if c in final.columns]
final = final[cols].sort_values('수출전망증가율(%)', ascending=False, na_position='last').reset_index(drop=True)
print(f"매칭 성공 {final['수출전망증가율(%)'].notna().sum()} / 전체 {len(final)}")
display(final.head(50))

매칭 성공 546 / 전체 622


,ticker,Name,hs_code_raw,hscode_6d,품목설명,past_12m,future_12m,수출전망증가율(%)
0,A063160,종근당바이오,2941309000,294130,(29류) 유기화학품,3.334214e+06,4.412651e+09,132244.53
1,A034020,두산중공업,8402110000,840211,(84류) 일반기계,7.800783e+06,1.084270e+09,13799.50
2,A306200,세아제강,7306292000,730629,유정용 케이싱·튜빙 강관(기타),9.493502e+08,4.011029e+10,4125.03
3,A004020,현대제철,7306292000,730629,유정용 케이싱·튜빙 강관(기타),9.493502e+08,4.011029e+10,4125.03
4,A004020,현대제철,7306291000,730629,유정용 케이싱·튜빙 강관(기타),9.493502e+08,4.011029e+10,4125.03
5,A011780,금호석유,2921519090,292151,(29류) 유기화학품,2.673821e+05,1.061052e+07,3868.30
6,A004020,현대제철,7216401000,721640,(72류) 철강,4.077009e+06,1.555439e+08,3715.15
7,A036560,영풍정밀,870530,870530,(87류) 자동차·부품,1.791921e+08,6.186273e+09,3352.31
8,A041440,에버다임,870530,870530,(87류) 자동차·부품,1.791921e+08,6.186273e+09,3352.31
9,A147830,제룡산업,850239,850239,(85류) 전기·전자기기,8.076327e+06,1.494662e+08,1750.67


In [9]:
final[final['Name'] == '금호석유']

,ticker,Name,hs_code_raw,hscode_6d,품목설명,past_12m,future_12m,수출전망증가율(%)
5,A011780,금호석유,2921519090,292151,(29류) 유기화학품,2.673821e+05,1.061052e+07,3868.30
255,A011780,금호석유,381239,381239,(38류) 각종 화학공업품,1.658970e+08,1.752329e+08,5.63
378,A011785,금호석유,4002110000,400211,(40류) 고무,6.702937e+07,6.698712e+07,-0.06
383,A011781,금호석유,4002190000,400219,"스티렌부타디엔고무(SBR,기타)",1.104423e+09,1.101662e+09,-0.25
397,A011784,금호석유,4002590000,400259,(40류) 고무,2.642291e+08,2.608654e+08,-1.27
463,A011782,금호석유,4002209000,400220,부타디엔고무(BR),6.264223e+08,5.939985e+08,-5.18
542,A011780,금호석유,4002510000,400251,합성고무-NBR 라텍스(니트릴 라텍스),5.773875e+08,1.055185e+08,-81.72
572,A011780,금호석유,2930904090,293090,(29류) 유기화학품,NaN,NaN,NaN
573,A011780,금호석유,3812301000,381230,(38류) 각종 화학공업품,NaN,NaN,NaN
621,A011783,금호석유,4002709000,400270,(40류) 고무,NaN,NaN,NaN


In [ ]:
# Excel 저장
import os
from datetime import date
SAVE_DIR = os.environ.get('ANALYSIS_DIR', os.getcwd())
os.makedirs(SAVE_DIR, exist_ok=True)
out_path = os.path.join(SAVE_DIR, f"기업주력상품_수출전망_{date.today():%Y-%m-%d}.xlsx")
final.to_excel(out_path, index=False)
print("저장 완료:", out_path)